# Week 15: RAG Supplies Evidence at Request Time

This notebook follows the reviewed Week 15 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. RAG Supplies Evidence at Request Time
2. Parsing Recovers Content from Source Files
3. Cleaning Must Preserve Provenance
4. Chunking Defines the Unit of Retrieval
5. Embeddings Place Chunks and Queries in One Vector Space
6. Cosine Similarity Compares Vector Direction
7. A Vector Store Connects Vectors to Evidence
8. Retrieval Selects Top-k Evidence
9. Context Assembly Builds the Evidence Packet
10. Grounded Answers Need Verifiable Citations
11. A RAG Failure Has an Owning Stage
12. Guided Lab: Build RAG from Scratch

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. RAG Supplies Evidence at Request Time

**Retrieval-augmented generation (RAG)** finds relevant evidence and includes it in the model context before generation.

The two main phases are:

- **ingestion:** parse, clean, chunk, embed, and index documents;
- **query:** retrieve chunks, assemble context, generate, validate, and cite.

RAG can use current or private documents without changing model weights. It does not guarantee that retrieval found the right evidence or that generation used it faithfully.

### Work it out first

Question: `What is the refund period?`

Retriever returns a policy chunk stating `Refund requests are accepted within 14 days.` The model answers `14 days` and cites the source section.

Without supporting text, the system should return insufficient evidence.

### Notebook bridge

The from-scratch notebook builds each RAG stage explicitly before using a framework.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
question -> retrieve evidence -> build context -> generate answer -> verify citations

Expected output:

```text
An answer supported by retrieved chunks or an explicit insufficient-evidence result.
```


## 2. Parsing Recovers Content from Source Files

**Document parsing** converts a source format into structured text and metadata.

A parser may need to recover:

- headings and paragraphs;
- tables;
- page numbers;
- lists and reading order;
- captions;
- document identifiers.

PDF is designed for visual layout, so extracted text may have broken order, missing tables, or repeated headers. Ingestion must inspect output rather than assuming successful file opening means correct extraction.

### Work it out first

Two-column PDF text may be extracted across columns:

`left line 1, right line 1, left line 2...`

The words exist, but reading order is corrupted. A retrieval result from that text may be misleading.

### Notebook bridge

The advanced RAG notebook begins with document parsing before embeddings.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
documents = parser.load("policy.pdf")
print(len(documents), documents[0].metadata)

Expected output:

```text
Extracted document units plus source and page metadata for inspection.
```


## 3. Cleaning Must Preserve Provenance

**Cleaning** removes extraction noise or normalizes representation without changing meaning.

**Provenance** records where every block came from:

- source ID and version;
- page;
- section heading;
- stable URL or file path;
- ingestion time;
- access-control metadata.

Keep raw extracted text or a reference to it. If cleaning changes content, the transformation should be reproducible and inspectable.

### Work it out first

Repeated header `DS Academy Policy 2026` appears on every page.

Removing exact repeated headers is reasonable. Removing every line containing `Policy` may delete real content. Record the cleaning rule and compare before/after counts.

### Notebook bridge

Learners add metadata before chunk storage, not after answering.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
chunk.metadata = {
    "source_id": "policy-2026-v3",
    "page": 7,
    "section": "Refunds",
}

Expected output:

```text
Each future retrieval result can identify its exact source location.
```


## 4. Chunking Defines the Unit of Retrieval

A **chunk** is one retrievable evidence unit.

Chunking choices affect:

- whether a complete idea stays together;
- embedding specificity;
- retrieval recall;
- prompt size;
- citation precision.

**Overlap** repeats boundary text in adjacent chunks. It can preserve context but increases storage and duplicate retrieval.

Prefer document structure such as headings and paragraphs before fixed character cuts.

### Work it out first

A 900-token section becomes:

- three chunks of 300 tokens with no overlap; or
- four chunks near 300 tokens with 50-token overlap.

The second keeps boundary context but may retrieve repeated statements.

### Notebook bridge

Learners compare two chunking strategies on the same retrieval questions.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
chunks = splitter.split_documents(documents)

Expected output:

```text
A list of chunks retaining source metadata and bounded text lengths.
```


## 5. Embeddings Place Chunks and Queries in One Vector Space

An **embedding model** maps text into a fixed-length vector.

For vector retrieval:

1. embed every document chunk;
2. store chunk vector plus text and metadata;
3. embed the user query with the same compatible model;
4. compare the query vector with chunk vectors;
5. return the highest-ranked allowed chunks.

Changing embedding model usually requires rebuilding the stored index.

### Work it out first

Three chunks become 768-dimensional vectors. The query also becomes a 768-dimensional vector.

The shapes match, allowing similarity comparison. A 384-dimensional query from another model cannot be compared directly.

### Notebook bridge

Both RAG notebooks create document and query embeddings before search.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
chunk_vectors = embedder.embed_documents(chunk_texts)
query_vector = embedder.embed_query(question)
print(len(chunk_vectors[0]), len(query_vector))

Expected output:

```text
Matching embedding dimensions.
```


## 6. Cosine Similarity Compares Vector Direction

Cosine similarity:

`cos(a,b) = (a · b) / (||a|| ||b||)`

- `a · b`: dot product
- `||a||`: length of vector `a`
- denominator normalizes vector magnitudes
- result is commonly between `-1` and `1`

Higher cosine similarity means vector directions are more aligned. Whether higher means more relevant must be tested for the task.

### Work it out first

`a=[1,0]`, `b=[1,1]`

Dot product `= 1`  
`||a|| = 1`  
`||b|| = sqrt(2)`  
Cosine `= 1/sqrt(2) ≈ 0.707`

### Notebook bridge

The from-scratch notebook calculates similarity before relying on a vector database.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import numpy as np

a = np.array([1.0, 0.0])
b = np.array([1.0, 1.0])
similarity = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(round(similarity, 3))

Expected output:

```text
0.707
```


## 7. A Vector Store Connects Vectors to Evidence

A **vector store** keeps:

- embedding vector;
- chunk text or reference;
- source metadata;
- access metadata;
- stable chunk ID.

An exact search compares all vectors. Approximate indexes trade some recall for lower latency at scale.

Apply authorization filters before or during retrieval. Filtering after retrieval can leak whether protected documents exist and waste candidate slots.

### Work it out first

User belongs to tenant `A`.

Search condition:

`tenant_id = A AND document_status = published`

Only vectors satisfying both conditions may compete for top-k ranking.

### Notebook bridge

Learners store text and metadata together and test a denied cross-tenant query.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
results = vector_store.similarity_search(
    question,
    k=5,
    filter={"tenant_id": tenant_id, "status": "published"},
)

Expected output:

```text
Up to five authorized chunks with text, score, and provenance.
```


## 8. Retrieval Selects Top-k Evidence

A **retriever** turns a question into an ordered list of chunks.

`top-k` is the maximum number returned.

Larger `k` may improve coverage but adds irrelevant or duplicated context. Smaller `k` is focused but may miss necessary evidence.

Inspect:

- query;
- returned text;
- scores and score direction;
- source diversity;
- metadata filters;
- whether the answer requires multiple chunks.

### Work it out first

Relevant policy appears at rank `4`.

With `k=3`, it is missed.  
With `k=5`, it is included alongside two irrelevant chunks.

The choice must be evaluated across a query dataset, not one example.

### Notebook bridge

The notebook's nearest-neighbour results become an explicit retrieval report.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
for rank, document in enumerate(retriever.invoke(question), start=1):
    print(rank, document.metadata, document.page_content[:120])

Expected output:

```text
An inspectable ranked list with source metadata and text previews.
```


## 9. Context Assembly Builds the Evidence Packet

**Context assembly** converts retrieved chunks into a model-ready evidence packet.

It should:

- remove duplicates;
- preserve source IDs and locations;
- order chunks deliberately;
- separate evidence from instructions;
- trim to the token budget;
- include only authorized content;
- avoid splitting a citation from its text.

The model should be told that retrieved content is untrusted evidence, not new instructions.

### Work it out first

Five chunks consume `3,200` tokens, but evidence budget is `2,000`.

After deduplication, four chunks consume `2,500`. The assembler selects the highest-value set under `2,000` while retaining at least one chunk from each required source.

### Notebook bridge

Learners build context explicitly before the generation call.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
context = "\n\n".join(
    f"[{doc.metadata['source_id']} p.{doc.metadata['page']}]\n{doc.page_content}"
    for doc in selected_docs
)

Expected output:

```text
A labelled context packet whose citations can map back to retrieved documents.
```


## 10. Grounded Answers Need Verifiable Citations

An answer is **grounded** when its factual claims are supported by the supplied evidence.

A **citation** identifies the exact source and location supporting a claim.

Validation should check:

- cited source was retrieved;
- cited passage supports the nearby claim;
- no unsupported claims were added;
- quotation and numbers match;
- insufficient evidence produces refusal or qualification.

A citation string generated by the model is not proof by itself.

### Work it out first

Evidence: `[policy-v3 p.7] Refund requests are accepted within 14 days.`

Supported answer: `Refund requests are accepted within 14 days [policy-v3 p.7].`

Unsupported addition: `A full refund is guaranteed.` The passage does not support that claim.

### Notebook bridge

Learners add stable chunk IDs to the answer schema and verify them.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
valid_ids = {doc.metadata["chunk_id"] for doc in selected_docs}
assert set(answer.citation_ids) <= valid_ids

Expected output:

```text
The citation-ID check passes only for chunks actually supplied to the model.
```


## 11. A RAG Failure Has an Owning Stage

Common failure boundaries:

- parsing lost text;
- cleaning changed meaning;
- chunk boundary split evidence;
- embedding failed on domain terms;
- filter excluded the source;
- top-k missed the relevant chunk;
- context assembly removed or duplicated evidence;
- generator ignored evidence;
- citation did not support the claim.

Fix the owning stage. Prompt changes cannot restore text that ingestion lost.

### Work it out first

Answer misses the refund period.

Diagnosis:

1. source PDF contains `14 days`;
2. parsed text contains it;
3. chunk index contains it;
4. retriever ranks it `12`;
5. top-k is `5`.

Owning failure is retrieval ranking, not generation.

### Notebook bridge

The lab records intermediate artifacts for each query.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
debug_record = {
    "query": question,
    "retrieved_chunk_ids": [d.metadata["chunk_id"] for d in docs],
    "answer": answer.model_dump(),
}

Expected output:

```text
One trace connecting query, evidence, answer, and citations.
```


## 12. Guided Lab: Build RAG from Scratch

Build a small policy assistant:

1. parse and visually verify source pages;
2. preserve source, page, section, and access metadata;
3. compare two chunking strategies;
4. embed chunks and queries with one model;
5. calculate one cosine similarity by hand;
6. store vectors and metadata;
7. retrieve authorized top-k chunks;
8. inspect ranks before generation;
9. assemble deduplicated labelled context;
10. return structured answer and citation IDs;
11. verify citation support;
12. refuse an unsupported question.

### Work it out first

Required test:

Question whose answer exists, question requiring two chunks, exact identifier query, unsupported question, and unauthorized-document query.

Each failure must identify its owning stage.

### Notebook bridge

Complete `17.rag-from-scratch.ipynb` and `06.build-rag.ipynb`.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
answer = rag(question, user_context)
assert all(cid in retrieved_ids for cid in answer.citation_ids)

Expected output:

```text
A grounded answer with valid citations or a typed insufficient/denied result.
```


## Guided lab

Build a small policy assistant:

1. parse and visually verify source pages;
2. preserve source, page, section, and access metadata;
3. compare two chunking strategies;
4. embed chunks and queries with one model;
5. calculate one cosine similarity by hand;
6. store vectors and metadata;
7. retrieve authorized top-k chunks;
8. inspect ranks before generation;
9. assemble deduplicated labelled context;
10. return structured answer and citation IDs;
11. verify citation support;
12. refuse an unsupported question.

### Reference result

Required test:

Question whose answer exists, question requiring two chunks, exact identifier query, unsupported question, and unauthorized-document query.

Each failure must identify its owning stage.


In [ ]:
# Guided lab workspace: Week 15
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://github.com/curiousily/AI-Bootcamp/blob/master/17.rag-from-scratch.ipynb>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/06.build-rag.ipynb>
- <https://docs.langchain.com/oss/python/integrations/document_loaders>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/07.advanced-rag-with-llama-3-in-langchain.ipynb>
- <https://python.langchain.com/docs/concepts/documents/>
- <https://www.w3.org/TR/prov-overview/>
- <https://docs.langchain.com/oss/python/integrations/splitters>
- <https://docs.langchain.com/oss/python/integrations/text_embedding>
- <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html>
- <https://docs.langchain.com/oss/python/integrations/vectorstores>
- <https://github.com/pgvector/pgvector>
- <https://python.langchain.com/docs/concepts/retrievers/>
- <https://docs.langchain.com/oss/python/langchain/rag>
- <https://docs.langchain.com/langsmith/evaluate-rag-tutorial>
- <https://docs.langchain.com/langsmith/evaluation-concepts>